In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/Users/katriellayam/Downloads/IPIP300-SCORES.csv')  # adjust filename to match your download
df.head()

print(df.columns.tolist())

In [ ]:
# Keep only Big Five scores

ocean_cols = [
    'openness',
    'conscientiousness',
    'extraversion',
    'agreeableness',
    'neuroticism'
]

df_ocean = df[ocean_cols].dropna()

scaler = StandardScaler()


In [ ]:
train_df, test_df = train_test_split(df_ocean, test_size=0.2, random_state=42)


In [ ]:
# stanrdardize features on test and training sets
x_train = scaler.fit_transform(train_df)
x_test = scaler.transform(test_df)


In [ ]:
train_df.head()


In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(x_train)

plt.figure(figsize=(10, 6))
plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1],
            s=5,          # slightly bigger points
            alpha=0.2,    # more transparent to show density
            color='steelblue')

plt.title('Training Data — PCA Projection (2D)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.tight_layout()
plt.savefig('visual_train_data.png', dpi=150)
plt.show()

In [ ]:
# define K-means function
def kmeans(X, k, max_iters=300, tol=1e-4, random_state=42):
    # create a random number generator to make reproducible results
    rand_numb = np.random.default_rng(random_state)

    # pick first centroid
    # randomly genereate an integer between 0 and length of the dataset (n-1)
    first_index = rand_numb.integers(0, len(X))
    centroids = [X[first_index]] # start centroid list with randomly generated integer

    # for loop until we have k centroids
    for _ in range(k - 1):
        # compute the squared distance to the nearest centroid for each point
        # take minimum distance for every point from centroid c
        dists = np.array([
            min(np.sum((x - c) ** 2) for c in centroids)
            for x in X
        ])

        # convert distances to be probabilities
        probs = dists / dists.sum()

        # determine the next centroid
        # draw a random number and find its position through a cumulative sum of the
        # probailities created
        next_index = np.searchsorted(np.cumsum(probs), rand_numb.random())

        # add that point to list of centroids
        centroids.append(X[next_index])

    # convert list of centroids into a numpy array
    centroids = np.array(centroids)

    # create an array of labels that will track which cluster each point belongs to
    labels = np.zeros(len(X), dtype=int) # start with zero array

    # loop up to the max number of iterations set by parameter
    for iteration in range(max_iters):
        # compute the difference between every point and every centroid
        diffs = X[:, np.newaxis, :] - centroids[np.newaxis, :, :]

        # square the differences calculated and sum them (squared Euclidean distance)
        dists = np.sum(diffs ** 2, axis=2)

        # for each point, find the index of the smallest distance
        # this index is the cluster ID for that point
        new_labels = np.argmin(dists, axis=1)

        # recalculate the centroids so that it is the mean of the
        # points that were assigned to it
        new_centroids = np.array([
            X[new_labels == j].mean(axis=0)
            if np.any(new_labels == j)
            else centroids[j]
            for j in range(k)
                                 ])

        # measure how far each centroid moved from the previous iteration

        # subtract old centroid from new centroid to get the distance it moved
        # calculated by squaring the sum across centroid coordinates and taking its square root
        # get the max of these centroid movements across all k centroids
        shift = np.sqrt(np.sum((new_centroids - centroids) ** 2, axis=1)).max()

        # update labels and centroids for next iteration
        labels = new_labels
        centroids = new_centroids

        # if the centroid movement is less than the tol, then the algorithm has converged
        # the iterative process of updating clusters reached stability
        if shift < tol:
            print(f" K-Means converged at iteration {iteration + 1}")
            break

    # case for when we hit the max number of iterations established
    else:
        print(f" K-Means hit max_iters ({max_iters})")

    # compute inertia, which is the total sum of squared distances from each point to its centroid
    inertia = sum(
        np.sum((X[labels == j] - centroids[j]) ** 2)
        for j in range(k)
    )

    # return the labels, centroids, and inertia
    return labels, centroids, inertia


In [ ]:
# USED LATER FOR TESTING SET
# define function kmeans_predict which assigns each point in X to the nearest centroid
def kmeans_predict(X, centroids):
    # compute difference between each test point and each centroid
    diffs = X[:, np.newaxis, :] - centroids[np.newaxis, :, :]

    # square the differences calculated and sum them
    dists = np.sum(diffs ** 2, axis=2)

    # for each test point return the cluster ID
    return np.argmin(dists, axis=1)


In [ ]:
# have K = 6
K = 6

print(f"\nTraining K-Means with k={K}...")

# run our kmeans algorithm with the chosen k and training data
km_labels, km_centroids, km_inertia = kmeans(x_train, k=K, random_state=42)

In [ ]:
# project to 2D using PCA
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(x_train)

# Project centroids to same PCA space
centroids_pca = pca.transform(km_centroids)

colors = plt.cm.tab10(np.linspace(0, 1, K))

plt.figure(figsize=(11, 7))

for j in range(K):
    mask = km_labels == j
    plt.scatter(X_train_pca[mask, 0],
                X_train_pca[mask, 1],
                s=3, color=colors[j],
                alpha=0.3, label=f"Cluster {j}")

# Plot centroids
plt.scatter(centroids_pca[:, 0],
            centroids_pca[:, 1],
            s=200, c='black',
            marker='X', zorder=5,
            label='Centroids')

plt.title(f'K-Means Clusters — PCA Projection (k={K})')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(markerscale=1, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('kmeans_train.png', dpi=150)
plt.show()

In [ ]:
# use trained centroids to predict cluster label for test dataset
km_test_labels = kmeans_predict(x_test, km_centroids)

# project test data to same PCA space (use the pca object already fitted on training data)
X_test_pca = pca.transform(x_test)
centroids_pca = pca.transform(km_centroids)

# visualize test clusters
plt.figure(figsize=(11, 7))

# plot each cluster
for j in range(K):
    mask = km_test_labels == j
    plt.scatter(X_test_pca[mask, 0], X_test_pca[mask, 1],
                s=4, color=colors[j], alpha=0.4, label=f"Cluster {j}")

# plot centroids
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            s=200, c='black', marker='X', zorder=5, label='Centroids')

plt.title(f'K-Means Test Clusters — PCA Projection (k={K})')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(markerscale=1, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('kmeans_test.png', dpi=150)
plt.show()